# **CLIP(Vision Transformer)**

## 1.환경준비

### (1)라이브러리 로딩

In [ ]:
import os, csv
import torch
from torchvision.datasets import CocoDetection
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

### (2) 디바이스 설정

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2.CLIP 사용해보기

### (1) 모델 다운로드

In [ ]:
# 모델 다운로드
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)

### (2) 모델 사용

#### 1)이미지와 문장 준비

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 dog.jpg 선택

In [ ]:
img_path = "dog.jpg"
image = Image.open(img_path).convert("RGB")

# 캡션 후보 (텍스트 프롬프트)
texts = [
    "a dog playing in the park",
    "a cat sleeping on the sofa",
    "a person riding a bicycle",
    "a bowl of ramen on the table"
]

#### 2)전처리 및 유사도 계산

In [ ]:
with torch.no_grad():
    inputs = processor(text=texts, images=image, return_tensors="pt", padding=True).to(device)
    outputs = model(**inputs)
    # CLIP은 이미지/텍스트 임베딩을 같은 공간으로 보냄
    image_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
    text_embeds  = outputs.text_embeds  / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)

    # 코사인 유사도(배치: 1 x N문장) -> torch.Size([1, len(texts)])
    sims = image_embeds @ text_embeds.T
    best_idx = sims.argmax(dim=1).item()

print(f"이미지와 가장 잘 어울리는 문장: {texts[best_idx]}")
print("유사도 점수들:", sims.squeeze(0).tolist())

In [ ]:
text_embeds

#### 3)제로샷 분류

In [ ]:
# 제로샷 분류(라벨 후보를 문장 프롬프트로 만들어 비교)
# 제대로 하려면 라벨을 프롬프트 템플릿에 끼워 넣는 것이 일반적 (예: "a photo of a {label}")
labels = ["dog", "cat", "bicycle", "ramen"]
label_prompts = [f"a photo of a {c}" for c in labels]

with torch.no_grad():
    inputs = processor(text=label_prompts, images=image, return_tensors="pt", padding=True).to(device)
    outputs = model(**inputs)
    pred_img = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
    pred_txt = outputs.text_embeds  / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
    pred_sims = (pred_img @ pred_txt.T).squeeze(0)  # shape: [num_labels]
    pred_idx = int(torch.argmax(pred_sims).item())

print(f"제로샷 분류 결과: {labels[pred_idx]}")
for i, (lab, sc) in enumerate(zip(labels, pred_sims.tolist())):
    print(f"- {lab:8s}: {sc:.4f}")

### (3) 실습
* 원하는 이미지와 문장을 준비 합니다.
    * 이미지는 사물, 동물
    * 문장은 최소 3~5개 영문으로 준비

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
img_path = "your image.jpg"  # 준비한 이미지
image = Image.open(img_path).convert("RGB")

# 캡션 후보 (텍스트 프롬프트)
texts = [    ]

* 이미지와 문장 간 유사도 계산

* 제로샷 분류
    * 이미지의 정답을 포함한 레이블 구성
    * 이미지 제로샷 분류 수행

In [ ]:
labels = [ ]
label_prompts = [f"a photo of a {c}" for c in labels]




## 3.이미지 태깅(레이블링)

* CLIP을 이용해서 이미지에 태그를 달아 봅시다.

### (1) 데이터 준비

* cocodataset2017 소개
    * 규모: 약 33만 장의 이미지와 250만 개 이상의 객체 인스턴스 포함
    * 클래스: 80개 객체 카테고리(사람, 동물, 차량, 가전제품, 음식 등 일상적인 사물)
    * 라벨: 객체 경계 상자(bounding box), 세그멘테이션 마스크, 키포인트(예: 사람 관절 위치), 자연어 캡션(이미지 설명 문장) 제공
    
* 데이터 다운로드 및 압축 풀기(약 2~3분 소요)

In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

!unzip val2017.zip -d ./coco/
!unzip annotations_trainval2017.zip -d ./coco/

* 데이터 100건만 샘플링해서 태깅을 수행해 봅니다.

In [ ]:
# 경로 설정
coco_root = "./coco/val2017"
ann_file  = "./coco/annotations/instances_val2017.json"

# CocoDetection 로드
dataset = CocoDetection(root=coco_root, annFile=ann_file)

In [ ]:
# 100장만 사용
subset = [dataset[i] for i in range(100)]

In [ ]:
i = 30
img, ann = subset[i]
plt.imshow(img)
plt.axis("off")
plt.title("COCO Sample")
plt.show()

### (2) 모델 준비

In [ ]:
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)

In [ ]:
# 환경
top_k = 5          # 이미지당 태그 최대 개수
thr = 0.01         # 과태깅 방지 임계값. None이면 미사용

In [ ]:
# 클래스 레이블 준비(태그)
coco80 = [
    "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light",
    "fire hydrant","stop sign","parking meter","bench","bird","cat","dog","horse","sheep","cow",
    "elephant","bear","zebra","giraffe","backpack","umbrella","handbag","tie","suitcase","frisbee",
    "skis","snowboard","sports ball","kite","baseball bat","baseball glove","skateboard","surfboard","tennis racket","bottle",
    "wine glass","cup","fork","knife","spoon","bowl","banana","apple","sandwich","orange",
    "broccoli","carrot","hot dog","pizza","donut","cake","chair","couch","potted plant","bed",
    "dining table","toilet","tv","laptop","mouse","remote","keyboard","cell phone","microwave","oven",
    "toaster","sink","refrigerator","book","clock","vase","scissors","teddy bear","hair drier","toothbrush"
]


In [ ]:
prompts = [f"A photo of a {l}" for l in coco80]

### (3) 태깅

In [ ]:
@torch.no_grad()  # 데코레이터. 함수 안에서 실행되는 모든 연산이 autograd(자동 미분)을 기록하지 않음
def tag_image(img: Image.Image, top_k=5):
    inputs = processor(text=prompts, images=img, return_tensors="pt", padding=True).to(device)
    out = model(**inputs)
    sims = out.logits_per_image.softmax(dim=1)[0].cpu()  # [80]
    vals, idxs = torch.topk(sims, k=top_k)
    tags = [(coco80[i], float(vals[j])) for j, i in enumerate(idxs)]
    return tags

In [ ]:
os.makedirs("coco_out", exist_ok=True)
csv_path = "coco_out/coco_clip_tags_0_100.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    wr = csv.writer(f)
    wr.writerow(["idx", "file", "topk_tags", "scores"])
    for i, (img, ann) in enumerate(subset):
        # 태깅
        tags = tag_image(img, top_k=5)

        # 저장용 파일 경로
        out_path = f"coco_out/val_{i:03d}.jpg"
        img.save(out_path)

        # CSV 기록
        wr.writerow([i, out_path,
                     "|".join([t for t, _ in tags]),
                     "|".join([f"{s:.4f}" for _, s in tags])])

        # 10장 정도만 시각화 예시
        if i < 10:
            plt.imshow(img)
            plt.axis("off")
            title = ", ".join([f"{t}:{s:.2f}" for t, s in tags])
            plt.title(title, fontsize=9)
            plt.show()

print(f"완료: {csv_path} (썸네일은 coco_out/에 저장됨)")

### (4) 실습

* 다양한 object가 있는 이미지를 업로드 합니다.

In [ ]:
from google.colab import files
uploaded = files.upload()

img_path = "your_image.jpg"
img = Image.open(img_path).convert("RGB")

* tag_image 함수를 사용하여 태깅을 수행해 봅시다.(top_k, thr 값 조정)

In [ ]:
import matplotlib.pyplot as plt

top_k =     # 상위 태그 개수
thr   =     # 낮은 점수 걸러내기. None이면 사용 안 함

tags = tag_image(   )

plt.imshow(img)
plt.axis("off")
plt.title(", ".join([f"{t}:{s:.2f}" for t, s in tags]), fontsize=9)
plt.show()

print("Top tags:")
for t, s in tags:
    print(f"- {t:15s} : {s:.4f}")